# 01 — Exploratory Data Analysis

Goal: understand the data before modelling — class balance, lengths, style/formatting cues, topics (domains), and how
the **test set differs from the training set** (the competition says it contains unseen styles and generators).

Run from the repo root's `notebooks/` folder with the `cs-aml` kernel. Everything here is read-only; nothing is written to disk.

In [ ]:
import sys, re, collections
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.features import stylometric_features, STYLO_NAMES

pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

D = ROOT / "data"
train = pd.read_csv(D / "train.csv")
test = pd.read_csv(D / "test.csv")
unl = pd.read_csv(D / "unlabeled.csv")
starter = pd.read_csv(D / "train_starter_2000.csv")
sample_sub = pd.read_csv(D / "sample_submission.csv")

# One combined frame with a 'group' column makes comparisons easy
allx = pd.concat([
    train.assign(group=np.where(train.label == 1, "train_AI", "train_human")),
    unl.assign(group="unlabeled", label=np.nan),
    test.assign(group="test", label=np.nan),
], ignore_index=True)
GROUPS = ["train_human", "train_AI", "unlabeled", "test"]
COLORS = {"train_human": "#2a78b5", "train_AI": "#d9822b", "unlabeled": "#8a8a8a", "test": "#3a9e6e"}

## 1. Basic facts

In [ ]:
for name, df in [("train", train), ("unlabeled", unl), ("test", test), ("starter", starter), ("sample_sub", sample_sub)]:
    print(f"{name:11s} shape={df.shape}  columns={list(df.columns)}  null text={df['text'].isna().sum() if 'text' in df else '-'}")
print("\nLabel balance (train):", train.label.value_counts().to_dict())
print("Starter is a subset of train:", starter.id.isin(train.id).all())
print("Duplicate texts within train:", train.text.duplicated().sum())
print("Exact text overlap train∩test:", len(set(train.text) & set(test.text)),
      "| train∩unlabeled:", len(set(train.text) & set(unl.text)))
print("ID ranges  train:", (train.id.min(), train.id.max()), " unlabeled:", (unl.id.min(), unl.id.max()), " test:", (test.id.min(), test.id.max()))

## 2. Length distributions

In [ ]:
allx["n_words"] = allx.text.str.split().str.len()
allx["n_chars"] = allx.text.str.len()
display(allx.groupby("group")[["n_words", "n_chars"]].describe().round(1).loc[GROUPS])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(70, 355, 40)
for g in ["train_human", "train_AI"]:
    ax[0].hist(allx.loc[allx.group == g, "n_words"], bins=bins, alpha=.55, label=g, color=COLORS[g], density=True)
for g in ["unlabeled", "test"]:
    ax[1].hist(allx.loc[allx.group == g, "n_words"], bins=bins, alpha=.55, label=g, color=COLORS[g], density=True)
ax[1].hist(allx.loc[allx.group.str.startswith("train"), "n_words"], bins=bins, histtype="step", color="k", label="train (all)", density=True)
for a in ax: a.set_xlabel("words per passage"); a.legend()
ax[0].set_title("Train: human vs AI"); ax[1].set_title("Unlabeled / test vs train")
plt.show()

## 3. Formatting cues

Formatting can be a very strong (and possibly *fragile*) signal: it reflects how each source was scraped/generated
rather than who wrote it. Compare the rates in train with test — if a cue is common in train-AI but distributed differently
in test, a model that leans on it may not generalise.

In [ ]:
cues = {
    "has blank line (\\n\\n)": r"\n\s*\n",
    "has any newline": r"\n",
    "has double space": r"  ",
    "has non-ASCII char": r"[^\x00-\x7f]",
    "has zero-width char": "[\u200b\u200c\u200d\ufeff]",
    "has Cyrillic char": "[\u0400-\u04ff]",
    "has '/' (e.g. 1/2 cup)": r"/",
    "has markdown (* or #) bullets": r"(?:^|\n)\s*(?:[*#-]|\d+\.)\s",
    "starts with quote": r'^\s*"',
    "ends without .!?\"": r'[^.!?"\')\s]\s*$',
}
rate = pd.DataFrame({name: allx.groupby("group").text.apply(lambda s: s.str.contains(p, regex=True).mean())
                     for name, p in cues.items()}).T[GROUPS]
display((rate * 100).round(1).style.background_gradient(axis=1, cmap="Blues").format("{:.1f}%"))

## 4. Stylometric features

Per-passage statistics computed by `src/features.py` (the same features used by the `stylometric_gbm` model).
The table ranks each feature by its single-feature AUROC on train (0.5 = useless, further from 0.5 = more informative).

In [ ]:
F = pd.DataFrame(stylometric_features(allx.text.tolist()), columns=STYLO_NAMES)
F["group"] = allx.group.values
from sklearn.metrics import roc_auc_score
tr_mask = F.group.str.startswith("train")
ytr = (F.group[tr_mask] == "train_AI").astype(int)
auc = pd.Series({c: roc_auc_score(ytr, F.loc[tr_mask, c]) for c in STYLO_NAMES}).sort_values(key=lambda s: -abs(s - .5))
summary = F.groupby("group")[STYLO_NAMES].median().T[GROUPS]
summary.insert(0, "AUROC (AI>human)", auc)
display(summary.loc[auc.index].head(20).round(3))

In [ ]:
show = ["sent_len_mean", "burstiness", "ttr", "word_len_mean", "stopword_ratio", "contractions", "rep_trigram", "punct_,"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), show):
    data = [F.loc[F.group == g, c].clip(upper=F[c].quantile(.99)) for g in GROUPS]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True)
    for patch, g in zip(bp["boxes"], GROUPS): patch.set_facecolor(COLORS[g]); patch.set_alpha(.6)
    ax.set_xticks(range(1, 5), ["human", "AI", "unlab", "test"]); ax.set_title(f"{c}  (AUROC {auc[c]:.2f})")
plt.tight_layout(); plt.show()

## 5. Topics / domains

Cluster all passages by content (TF-IDF + k-means). This shows which kinds of text exist (recipes, abstracts, news, poems, …),
how balanced human/AI is inside each, and **whether test contains domains that are rare in train**.
`cv.scheme=cluster` in `train.py` uses the same idea to hold out whole topics during validation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
K = 10
vec = TfidfVectorizer(max_features=30000, min_df=3, stop_words="english", sublinear_tf=True)
Xall = vec.fit_transform(allx.text)
km = MiniBatchKMeans(K, random_state=0, n_init=5).fit(Xall)
allx["cluster"] = km.labels_
terms = np.array(vec.get_feature_names_out())
top_terms = {k: ", ".join(terms[np.argsort(km.cluster_centers_[k])[::-1][:8]]) for k in range(K)}

tab = pd.crosstab(allx.cluster, allx.group, normalize="columns")[GROUPS] * 100
tab["AI share in train"] = allx[allx.group.str.startswith("train")].groupby("cluster").label.mean() * 100
tab["test / train ratio"] = tab["test"] / ((tab["train_human"] + tab["train_AI"]) / 2)
tab["top terms"] = pd.Series(top_terms)
display(tab.round(1))

## 6. Most class-indicative words (log-odds with smoothing)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(1, 2), min_df=10, lowercase=True, token_pattern=r"(?u)\b\w+\b")
C = cv.fit_transform(train.text)
v = np.array(cv.get_feature_names_out())
ai, hu = np.asarray(C[train.label.values == 1].sum(0)).ravel() + 1, np.asarray(C[train.label.values == 0].sum(0)).ravel() + 1
lo = np.log(ai / ai.sum()) - np.log(hu / hu.sum())
order = np.argsort(lo)
print("Most AI-indicative:   ", ", ".join(v[order[::-1][:40]]))
print("\nMost human-indicative:", ", ".join(v[order[:40]]))

## 7. Clues about the generators

Telltale phrases hint at which kinds of models produced the AI text (chat-tuned assistants say "Sure, here's…",
older base models such as GPT-2 tend to loop/repeat). Comparing rates in train vs test hints at generators that are new in test.
This is useful if you want to generate extra AI training data with similar (or deliberately *different*) models.

In [ ]:
phrases = {
    "assistant opener (Sure/Certainly/Here's)": r"^\s*(?:Sure|Certainly|Of course|Absolutely|Here's|Here is)\b",
    "'As an AI'": r"\bas an ai\b",
    "delve / tapestry / testament": r"\b(?:delve|tapestry|testament to|intricate)\b",
    "In conclusion / Overall,": r"\b(?:In conclusion|Overall,|In summary)\b",
    "markdown bold **": r"\*\*",
    "Title: / Ingredients: header": r"(?:^|\n)(?:Title|Ingredients|Instructions|Directions):",
    "heavy repetition (rep_trigram>0.15)": None,
}
rows = {}
for name, p in phrases.items():
    if p is None:
        rows[name] = (F.rep_trigram > .15).groupby(F.group).mean()
    else:
        rows[name] = allx.text.str.contains(p, regex=True, case=False).groupby(allx.group).mean()
display((pd.DataFrame(rows).T[GROUPS] * 100).round(2))

In [ ]:
# Look at the most repetitive AI passages (typical of small base models with greedy decoding)
rep = allx.assign(rep=F.rep_trigram.values).query("group == 'train_AI'").sort_values("rep", ascending=False)
for t in rep.text.head(3): print(repr(t[:400]), "\n")

## 8. How different is test from train? (adversarial validation)

Train a classifier to tell *train* passages from *test* passages. AUROC ≈ 0.5 means the two look alike; higher means a
distribution shift. The train passages that look most "test-like" are a natural choice for a validation set that mimics test.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
both = pd.concat([train.text, test.text], ignore_index=True)
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]
Xa = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=100000).fit_transform(both)
p = cross_val_predict(LogisticRegression(C=1, max_iter=2000), Xa, is_test, cv=5, method="predict_proba")[:, 1]
print(f"Adversarial AUROC train-vs-test: {roc_auc_score(is_test, p):.3f}")
train["test_likeness"] = p[: len(train)]
print("\nMost test-like train passages:")
display(train.sort_values("test_likeness", ascending=False)[["id", "label", "test_likeness", "text"]].head(8))
plt.hist(train.test_likeness, bins=50); plt.xlabel("P(test | text) for train passages"); plt.title("Test-likeness of train passages"); plt.show()

## 9. Browse examples

In [ ]:
def show(group="train_AI", n=3, cluster=None, seed=None):
    d = allx[allx.group == group]
    if cluster is not None: d = d[d.cluster == cluster]
    for _, r in d.sample(n, random_state=seed).iterrows():
        print(f"--- id={r.id} group={r.group} cluster={r.cluster} words={r.n_words}")
        print(r.text[:700], "\n")

show("train_human", 2, seed=0)
show("train_AI", 2, seed=0)
show("test", 2, seed=0)